In [1]:
from collections import defaultdict, Counter
from typing import Any, Dict, List, Optional, Tuple

from datasets import load_dataset

import numpy as np

import polars as pl

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🛠️ Подготовка данных

In [2]:
format = 'sequential'
size = '50m'
events = 'listens'
# listens_data = load_dataset('yandex/yambda', data_dir=f'{format}/{size}', data_files=f'{events}.parquet')
# yambda_df = pl.from_arrow(listens_data['train'].data.table)
yambda_df = pl.read_parquet("/home/jovyan/yambda_sequential_50m/sequential/50m/listens.parquet")

In [3]:
def test_yambda_data_loading():
    assert isinstance(yambda_df, pl.DataFrame), 'yambda_df должен быть Polars DataFrame'
    assert yambda_df.shape == (9238, 6), f'Неправильный размер: {yambda_df.shape}'

    expected_cols = {'uid', 'timestamp', 'item_id', 'is_organic', 'played_ratio_pct', 'track_length_seconds'}
    assert set(yambda_df.columns) == expected_cols, f'Неправильные колонки: {yambda_df.columns}'

    assert yambda_df['item_id'].dtype == pl.List(pl.UInt32), 'item_id должен быть List[UInt32]'
    assert yambda_df['timestamp'].dtype == pl.List(pl.UInt32), 'timestamp должен быть List[UInt32]'

    assert yambda_df['item_id'].list.len().min() > 0, 'Есть пустые истории'

    print('✅ test_yambda_data_loading: OK')

test_yambda_data_loading()

✅ test_yambda_data_loading: OK


In [4]:
(yambda_df.head(), yambda_df.filter(yambda_df['timestamp'].list.len() != yambda_df['item_id'].list.len()))

(shape: (5, 6)
 ┌─────┬─────────────────────┬────────────┬─────────────┬─────────────────────┬─────────────────────┐
 │ uid ┆ timestamp           ┆ item_id    ┆ is_organic  ┆ played_ratio_pct    ┆ track_length_second │
 │ --- ┆ ---                 ┆ ---        ┆ ---         ┆ ---                 ┆ s                   │
 │ u32 ┆ list[u32]           ┆ list[u32]  ┆ list[u8]    ┆ list[u16]           ┆ ---                 │
 │     ┆                     ┆            ┆             ┆                     ┆ list[u32]           │
 ╞═════╪═════════════════════╪════════════╪═════════════╪═════════════════════╪═════════════════════╡
 │ 100 ┆ [39420, 39420, …    ┆ [8326270,  ┆ [0, 0, … 0] ┆ [100, 100, … 100]   ┆ [170, 105, … 165]   │
 │     ┆ 25966140]           ┆ 1441281, … ┆             ┆                     ┆                     │
 │     ┆                     ┆ 4734787]   ┆             ┆                     ┆                     │
 │ 200 ┆ [14329075,          ┆ [3285270,  ┆ [1, 1, … 1] ┆ [9, 28, …

In [7]:
yambda_df = (
    yambda_df
    .filter(pl.col('uid') % 200 == 0) # надо убрать
    .with_row_index('_idx')
    .explode([
        'timestamp',
        'item_id',
        'is_organic',
        'played_ratio_pct',
        'track_length_seconds',
    ])
    .filter(
        (pl.col('is_organic') == 0) &
        (pl.col('played_ratio_pct') >= 50)
    )
    .group_by(['_idx', 'uid'], maintain_order=True)
    .agg([
        pl.col('timestamp'),
        pl.col('item_id'),
    ])
    .drop('_idx')
)

ColumnNotFoundError: 'explode' on column: 'is_organic' is invalid

Schema at this point: Schema:
name: _idx, field: UInt32
name: uid, field: UInt32
name: timestamp, field: List(UInt32)
name: item_id, field: List(UInt32)


In [8]:
def test_yambda_filtering():
    assert yambda_df.shape[0] == 4289, \
        f'Неправильное количество пользователей: {yambda_df.shape[0]}'

    expected_columns = {'uid', 'timestamp', 'item_id'}
    actual_columns = set(yambda_df.columns)
    assert actual_columns == expected_columns, \
        f'Неправильные колонки. Ожидалось: {expected_columns}, получено: {actual_columns}'

    assert yambda_df['timestamp'].dtype == pl.List(pl.UInt32), \
        f"timestamp должен быть List[UInt32], получено: {yambda_df['timestamp'].dtype}"
    assert yambda_df['item_id'].dtype == pl.List(pl.UInt32), \
        f"item_id должен быть List[UInt32], получено: {yambda_df['item_id'].dtype}"

    seq_lengths = yambda_df['item_id'].list.len()
    assert seq_lengths.min() >= 1, \
        f'Минимальная длина последовательности должна быть >= 1, получено: {seq_lengths.min()}'
    assert seq_lengths.sum() == 7587469, \
        f'Общее количество событий неверно. Ожидалось: 7587469, получено: {seq_lengths.sum()}'

    unique_items = yambda_df.select('item_id').explode('item_id').unique().shape[0]
    assert unique_items == 304787, \
        f'Количество уникальных айтемов неверно. Ожидалось: 304787, получено: {unique_items}'

    print('✅ test_yambda_filtering: OK')

test_yambda_filtering()

✅ test_yambda_filtering: OK


In [12]:
import polars as pl
import pandas as pd
import pickle

# === 1. Загрузить оригинальные embeddings ===
embeddings_path = "/home/jovyan/yambda_embeddings/embeddings.parquet"
emb_df = pl.read_parquet(embeddings_path)

print(f"Оригинальные эмбеддинги: {emb_df.shape}")
print(f"Колонки: {emb_df.columns}")
print(emb_df.head())

Оригинальные эмбеддинги: (7721749, 3)
Колонки: ['item_id', 'embed', 'normalized_embed']
shape: (5, 3)
┌─────────┬─────────────────────────────────┬─────────────────────────────────┐
│ item_id ┆ embed                           ┆ normalized_embed                │
│ ---     ┆ ---                             ┆ ---                             │
│ u32     ┆ list[f64]                       ┆ list[f64]                       │
╞═════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 2       ┆ [-1.534035, -0.366767, … 0.999… ┆ [-0.064638, -0.015454, … 0.042… │
│ 3       ┆ [-3.761467, -1.068254, … -2.66… ┆ [-0.163937, -0.046558, … -0.11… │
│ 4       ┆ [2.445533, -2.523603, … -0.536… ┆ [0.076272, -0.078707, … -0.016… │
│ 5       ┆ [0.832846, 0.116125, … -1.4857… ┆ [0.03149, 0.004391, … -0.05617… │
│ 6       ┆ [-2.431483, -0.56872, … 0.0946… ┆ [-0.10345, -0.024197, … 0.0040… │
└─────────┴─────────────────────────────────┴─────────────────────────────────┘


In [13]:
valid_item_ids = set(emb_df['item_id'].to_list())
print(f"\nВалидные item_id: {len(valid_item_ids)}")
valid_ids_pl = pl.Series(list(valid_item_ids))

valid_item_ids = set(emb_df['item_id'].to_list())
valid_ids_pl = pl.Series(list(valid_item_ids))

yambda_df_filtered = (
    yambda_df
    .with_columns(
        pl.col("item_id").list.eval(
            pl.when(pl.element().is_in(valid_ids_pl))
              .then(pl.int_range(pl.len()))
              .otherwise(None)
        ).list.drop_nulls().alias("valid_indices")
    )
    .with_columns([
        pl.col("item_id").list.gather(pl.col("valid_indices")),
        pl.col("timestamp").list.gather(pl.col("valid_indices"))
    ])
    .drop("valid_indices")
    .filter(pl.col("item_id").list.len() > 0)
    .rename({"item_id": "item_ids", "timestamp": "timestamps"})
)
yambda_df_filtered = yambda_df_filtered.filter(yambda_df_filtered['item_ids'].list.len() >= 5)

print(f"Было строк: {yambda_df.shape[0]}")
print(f"Стало строк: {yambda_df_filtered.shape[0]}")


Валидные item_id: 7721749
Было строк: 4289
Стало строк: 4138


3️⃣ Получите все уникальные ID треков из датасета и создайте маппинг: старый_id - новый_id, где новый_id находится в диапазоне от 0 до N - 1.

Модели глубокого обучения требуют, чтобы категориальные признаки (в нашем случае ID треков) были представлены целыми числами в диапазоне от 0 до N-1, где N — количество уникальных треков. Датасет Yambda содержит оригинальные ID треков, которые могут быть разреженными (например, [100, 5000, 7, 12000, ...]) — это неэффективно для embedding-таблиц.

In [16]:
unique_items = (
    yambda_df_filtered
    .select('item_ids')
    .explode('item_ids')
    .unique()
    .sort('item_ids')
).with_row_index('new_item_ids')


item_mapping = dict(zip(unique_items['item_ids'], unique_items['new_item_ids']))


yambda_df_filtered = yambda_df_filtered.with_columns([
    pl.col('item_ids')
        .map_elements(
            lambda items: [item_mapping[item] for item in items],
            return_dtype=pl.List(pl.UInt32)
        )
        .alias('item_ids')
])

In [ ]:
def test_item_mapping():
    assert unique_items.shape == (292865, 2), f'Неправильный размер unique_items: {unique_items.shape}'
    assert set(unique_items.columns) == {'new_item_ids', 'item_ids'}, 'Неправильные колонки unique_items'

    assert len(item_mapping) == 292865, f'Неправильный размер item_mapping: {len(item_mapping)}'
    assert item_mapping[50] == 0 and item_mapping[175] == 1 and item_mapping[195] == 2, \
        'Неверные первые маппинги'

    new_ids = unique_items['new_item_ids']
    assert new_ids.min() == 0 and new_ids.max() == 292864, 'new_item_id должны быть в [0, 292865,]'

    all_ids = yambda_df_filtered.select('item_ids').explode('item_ids')['item_ids']
    assert all_ids.min() == 0 and all_ids.max() == 292864, 'item_id в yambda_df не обновлены'
    assert all_ids.n_unique() == 292865, 'Количество уникальных item_id изменилось'

    print('✅ test_item_mapping: OK')

test_item_mapping()

✅ test_item_mapping: OK


In [27]:
print(f"\nМаппинг использует: {len(item_mapping)} уникальных item_id")

# === 3. Переиндексировать embeddings используя существующий маппинг ===
emb_df_reindexed = emb_df.with_columns(
    pl.col('item_id')
    .map_elements(
        lambda x: item_mapping.get(x, None),
        return_dtype=pl.UInt32
    )
    .alias('new_item_id')
).filter(pl.col('new_item_id').is_not_null()).drop('item_id', 'embed').rename({'new_item_id': 'item_id', 'normalized_embed': 'embedding'})

print(f"Переиндексированные эмбеддинги: {emb_df_reindexed.shape}")
print(emb_df_reindexed.head())




Маппинг использует: 292865 уникальных item_id
Переиндексированные эмбеддинги: (292865, 2)
shape: (5, 2)
┌─────────────────────────────────┬─────────┐
│ embedding                       ┆ item_id │
│ ---                             ┆ ---     │
│ list[f64]                       ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ [-0.0526, 0.048672, … -0.04217… ┆ 0       │
│ [0.090222, -0.00718, … -0.0862… ┆ 1       │
│ [-0.00822, -0.057882, … 0.2188… ┆ 2       │
│ [-0.107289, -0.034719, … -0.02… ┆ 3       │
│ [0.012762, -0.043315, … 0.1494… ┆ 4       │
└─────────────────────────────────┴─────────┘


In [34]:
def test_emb_item_mapping():
    all_ids = emb_df_reindexed['item_id']
    assert all_ids.min() == 0 and all_ids.max() == 292864, 'item_id в yambda_df не обновлены'
    assert all_ids.n_unique() == 292865, 'Количество уникальных item_id изменилось'

    print('✅ test_item_mapping: OK')

test_emb_item_mapping()

✅ test_item_mapping: OK


In [35]:
embeddings_output_parquet_path = "/home/jovyan/IRec/sigir/yambda_data/yambda_embeddings_reindexed.parquet"
emb_df_reindexed.write_parquet(embeddings_output_parquet_path)
print(f"\n✓ Сохранены embeddings: {embeddings_output_parquet_path}")


✓ Сохранены embeddings: /home/jovyan/IRec/sigir/yambda_data/yambda_embeddings_reindexed.parquet


In [41]:
def test_integrity(df):
    bad_rows = df.filter(
        (pl.col("item_ids").list.len() != pl.col("timestamps").list.len()) | (pl.col("timestamps").list.len() < 5)
    )
    
    if bad_rows.height > 0:
        print(f"ОШИБКА: {bad_rows.height} строк рассинхронизированы!")
        raise ValueError("Рассинхрон массивов!")
    
    print(f"Тест пройден: все {df.height} строк синхронизированы")

test_integrity(yambda_df_filtered)

Тест пройден: все 4138 строк синхронизированы


In [42]:
yambda_output_parquet_path = "/home/jovyan/IRec/sigir/yambda_data/yambda_sequential_50m_filtered_reindexed.parquet"
yambda_df_filtered.write_parquet(yambda_output_parquet_path)
print(f"Сохранён filtered yambda_df: {yambda_output_parquet_path}")

Сохранён filtered yambda_df: /home/jovyan/IRec/sigir/yambda_data/yambda_sequential_50m_filtered_reindexed.parquet


In [43]:
import json
mapping_output_path = "/home/jovyan/IRec/sigir/yambda_data/old_to_new_item_id_mapping.json"

with open(mapping_output_path, 'w') as f:
    json.dump({str(k): v for k, v in item_mapping.items()}, f, indent=2)

print(f"Сохранён маппинг: {mapping_output_path}")

Сохранён маппинг: /home/jovyan/IRec/sigir/yambda_data/old_to_new_item_id_mapping.json


In [44]:
yambda_df_filtered.head(10)

uid,timestamps,item_ids
u32,list[u32],list[u32]
600,"[1329190, 1329405, … 25997540]","[252026, 58171, … 201909]"
800,"[121100, 121290, … 25977310]","[20844, 198210, … 60455]"
1000,"[11335730, 11335925, … 25972225]","[46643, 57592, … 95670]"
1400,"[280570, 280735, … 25993315]","[4634, 213798, … 104891]"
1600,"[899275, 930305, … 25941890]","[223933, 154424, … 104876]"
2000,"[18814620, 18828965, … 25225145]","[137828, 138498, … 19072]"
2200,"[10053900, 10054120, … 25948025]","[4923, 231643, … 28122]"
2400,"[14246260, 14246390, … 25999860]","[157350, 217652, … 75038]"
2600,"[6089640, 6089915, … 25951140]","[9426, 202953, … 140393]"
